# Trabalho 4 — Classificação de atributos de áudio com SVM

Referência de estrutura (one-shot): [`trabalho/trabalho_parte_3.ipynb`](../../trabalho/trabalho_parte_3.ipynb).
Adaptamos carregamento, prevenção de vazamento, split, pré-processamento, `LinearSVC`
balanceado, tuning com recall e avaliação. Aqui treinamos **três classificadores binários
independentes**, nesta ordem: `cansaco`, `falta_ar`, `doenca_respiratoria`.

**Fluxo:** 1. carregar e validar; 2. separar consultas; 3. analisar correlações no treino;
4. montar pipeline; 5. treinar SVM base e ajustar SVM por target; 6. avaliar;
7. exportar pipelines; 8. demonstrar recarga.

As entradas são os 88 atributos eGeMAPSv02/Functionals do openSMILE, por áudio inteiro (uma linha por gravação).
Os rótulos vieram da classificação fornecida dos arquivos. `doenca_respiratoria` representa,
nesta base, o grupo de tosse; não é confirmação de diagnóstico. Um resultado positivo
significa reconhecimento desse rótulo na amostra, sem validação de uso clínico.

## Preparação

Em `trabalho4`, instale `python -m pip install -r requirements.txt`
e abra este notebook com o kernel desse ambiente. Execute todas as células em ordem.
Mantenha `svm_audio.py` ao lado do notebook. Por padrão, o notebook extrai atributos dos WAVs completos em `inputs/training`
e grava `outputs/atributos_audio_inteiro.parquet`. Os caminhos podem ser ajustados
na próxima célula. Para reutilizar uma base de áudios inteiros, defina `PARQUET_ENTRADA`.
As saídas de uma execução anterior serão substituídas.


In [ ]:
from pathlib import Path
import hashlib
import importlib.metadata
import json
import platform
import shutil
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

# Parâmetros opcionais injetados pelo executor da parte 2.
RAIZ_AUDIO = Path(RAIZ_AUDIO_CONFIG) if 'RAIZ_AUDIO_CONFIG' in globals() else None
if RAIZ_AUDIO is None:
    candidatos = [
        base / relativo
        for base in [Path.cwd(), *Path.cwd().parents]
        for relativo in (Path('.'), Path('analise_audio'), Path('trabalho4/analise_audio'))
    ]
    RAIZ_AUDIO = next((p.resolve() for p in candidatos if (p / 'svm_audio.py').is_file()), None)
if RAIZ_AUDIO is None:
    raise FileNotFoundError('Defina RAIZ_AUDIO para a pasta que contém svm_audio.py.')
sys.path.insert(0, str(RAIZ_AUDIO))
from svm_audio import PreprocessamentoCorrelacao

PASTA_AUDIOS = Path(globals().get('PASTA_AUDIOS_CONFIG', RAIZ_AUDIO / 'inputs/training')).resolve()
EXTRAIR_AUDIOS = 'PARQUET_ENTRADA' not in globals()
PARQUET = Path(globals().get('PARQUET_ENTRADA', RAIZ_AUDIO / 'outputs/atributos_audio_inteiro.parquet')).resolve()
SAIDA = Path(globals().get('SAIDA_TREINAMENTO', RAIZ_AUDIO / 'outputs/svm')).resolve()
MODELOS = SAIDA / 'modelos'
FIGURAS = SAIDA / 'figuras'
for pasta in (SAIDA, MODELOS, FIGURAS):
    pasta.mkdir(parents=True, exist_ok=True)
TARGETS = ['cansaco', 'falta_ar', 'doenca_respiratoria']
METADADOS = ['arquivo', 'parte', 'inicio_s', 'fim_s']
SEED = 42
N_FOLDS = 4
sns.set_theme(style='whitegrid')
print('Entrada:', PARQUET)
print('Saídas:', SAIDA)


## 1 — Carregamento, integridade e distribuição dos targets

Cada linha contém os atributos calculados sobre um áudio inteiro (`arquivo`).
A extração usa `duracao=None`, sem recortes nem média de atributos de segmentos.
Bases externas devem seguir o mesmo contrato; múltiplas linhas por arquivo são rejeitadas.
Excluímos os três targets e todos os metadados das features: o nome do arquivo contém
explicitamente a classe e causaria vazamento. Infinitos serão tratados como ausentes pelo
pipeline; medianas e constantes serão aprendidas somente nos subconjuntos de treino.


In [ ]:
if EXTRAIR_AUDIOS:
    import opensmile
    from extrair_atributos_audio import extrair_atributos, listar_audios

    extrator = opensmile.Smile(
        feature_set=opensmile.FeatureSet.eGeMAPSv02,
        feature_level=opensmile.FeatureLevel.Functionals,
    )
    linhas = []
    for caminho in listar_audios(PASTA_AUDIOS):
        print(f'Extraindo áudio inteiro: {caminho.name}')
        linhas.extend(extrair_atributos(extrator, caminho, duracao=None))
    PARQUET.parent.mkdir(parents=True, exist_ok=True)
    pd.concat(linhas, ignore_index=True).to_parquet(PARQUET, index=False)

dados = pd.read_parquet(PARQUET)
obrigatorias = METADADOS + TARGETS
faltantes = sorted(set(obrigatorias) - set(dados.columns))
if faltantes:
    raise ValueError(f'Colunas obrigatórias ausentes: {faltantes}')
if dados.empty or dados.columns.duplicated().any():
    raise ValueError('Base vazia ou com nomes de colunas duplicados.')
if dados[obrigatorias].isna().any().any():
    raise ValueError('Há metadados ou rótulos ausentes.')
if not dados[TARGETS].isin([0, 1]).all().all():
    raise ValueError('Os três targets devem conter apenas 0 ou 1.')
if dados['arquivo'].duplicated().any():
    raise ValueError('Esperada uma linha por áudio inteiro. Reextraia os WAVs com duracao=None.')
if not (dados['parte'].eq(1) & dados['inicio_s'].eq(0) & dados['fim_s'].gt(0)).all():
    raise ValueError('A base deve conter áudios inteiros, iniciando em zero e com parte=1.')
if (dados[TARGETS].sum(axis=1) > 1).any():
    raise ValueError('Esta divisão pressupõe os quatro grupos exclusivos da seleção original.')
FEATURES = [c for c in dados.columns if c not in obrigatorias]
if len(FEATURES) != 88:
    raise ValueError(f'Esperados 88 atributos eGeMAPSv02; encontrados {len(FEATURES)}.')
if not all(pd.api.types.is_numeric_dtype(dados[c]) for c in FEATURES):
    raise ValueError('Todos os atributos acústicos devem ser numéricos.')
consultas = dados.groupby('arquivo', sort=True)[TARGETS].first()
consultas['estrato'] = consultas[TARGETS].astype(str).agg('_'.join, axis=1)
print(f'{len(dados)} áudios inteiros; {len(consultas)} consultas; {len(FEATURES)} atributos.')
distribuicao = pd.DataFrame({
    'consultas_positivas': consultas[TARGETS].sum(),
    'consultas_negativas': len(consultas) - consultas[TARGETS].sum(),
})
display(distribuicao)
ax = distribuicao[['consultas_negativas', 'consultas_positivas']].plot.bar(
    figsize=(9, 4), rot=0, title='Distribuição das classes por consulta'
)
ax.set_ylabel('Consultas')
plt.tight_layout()
plt.savefig(FIGURAS / 'distribuicao.png', dpi=140)
plt.show()


## 2 — Divisão treino/teste e validação por consulta

Dividimos os arquivos, estratificados pela combinação dos três rótulos, mantendo o
mesmo teste para os três problemas. Cada arquivo corresponde a uma linha de atributos
do áudio inteiro. Na base atual: 16 consultas de treino e 4 de teste (uma por categoria).

A validação cruzada usa quatro folds estratificados por consulta. Nenhuma consulta
atravessa as fronteiras de treino/validação/teste. Não há identificador de pessoa;
se a mesma pessoa aparecer em consultas diferentes, será necessário agrupar por pessoa.


In [ ]:
arquivos_treino, arquivos_teste = train_test_split(
    consultas.index.to_numpy(), test_size=0.20, random_state=SEED,
    stratify=consultas['estrato'],
)
mascara_treino = dados['arquivo'].isin(arquivos_treino)
mascara_teste = dados['arquivo'].isin(arquivos_teste)
assert set(arquivos_treino).isdisjoint(arquivos_teste)
assert (mascara_treino ^ mascara_teste).all()
X_train = dados.loc[mascara_treino, FEATURES].reset_index(drop=True)
X_test = dados.loc[mascara_teste, FEATURES].reset_index(drop=True)
Y_train = dados.loc[mascara_treino, TARGETS].reset_index(drop=True)
Y_test = dados.loc[mascara_teste, TARGETS].reset_index(drop=True)
grupos_treino = dados.loc[mascara_treino, 'arquivo'].reset_index(drop=True)
grupos_teste = dados.loc[mascara_teste, 'arquivo'].reset_index(drop=True)
consultas_treino = consultas.loc[sorted(arquivos_treino)]
if consultas_treino['estrato'].value_counts().min() < N_FOLDS:
    raise ValueError('São necessárias pelo menos quatro consultas por estrato no treino.')
cv = []
fold_por_arquivo = {}
separador = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
for fold, (i_treino, i_val) in enumerate(
    separador.split(consultas_treino, consultas_treino['estrato']), start=1
):
    g_treino = consultas_treino.index[i_treino]
    g_val = consultas_treino.index[i_val]
    idx_treino = np.flatnonzero(grupos_treino.isin(g_treino))
    idx_val = np.flatnonzero(grupos_treino.isin(g_val))
    assert set(g_treino).isdisjoint(g_val)
    assert set(g_val).isdisjoint(arquivos_teste)
    assert Y_train.iloc[idx_treino].nunique().eq(2).all()
    assert Y_train.iloc[idx_val].nunique().eq(2).all()
    cv.append((idx_treino, idx_val))
    fold_por_arquivo.update({arquivo: fold for arquivo in g_val})
assert Y_test.nunique().eq(2).all()
split = consultas.copy()
split['conjunto'] = np.where(split.index.isin(arquivos_treino), 'treino', 'teste')
split['fold_validacao'] = pd.Series(fold_por_arquivo).reindex(split.index).astype('Int64')
split.to_csv(SAIDA / 'divisao_consultas.csv')
display(split)
print('Áudios inteiros no treino:', len(X_train), '| teste:', len(X_test))
print('Infinitos/ausentes no treino:', X_train.replace([np.inf, -np.inf], np.nan).isna().sum().sum())


## 3 — Correlação entre atributos e com os targets (somente treino)

Calculamos Pearson entre atributos e entre cada atributo e os alvos binários
(equivalente à correlação ponto-bisserial para esses alvos). Usamos o sinal para
interpretar a associação e o valor absoluto para ordenar. A matriz completa é exportada;
os gráficos mostram os 20 atributos com maior associação absoluta a algum target.

O relatório exploratório imputa medianas do treino. A seleção efetiva **é refeita dentro
de cada fold**, sem reutilizar este ranking global na validação. Correlação não implica
causalidade; gravações diferentes não necessariamente representam pessoas diferentes.


In [ ]:
exploracao = X_train.replace([np.inf, -np.inf], np.nan)
exploracao = exploracao.fillna(exploracao.median().fillna(0.0))
variaveis = exploracao.loc[:, exploracao.nunique() > 1]
corr_features = variaveis.corr()
corr_targets = pd.DataFrame({t: variaveis.corrwith(Y_train[t]) for t in TARGETS})
corr_features.to_csv(SAIDA / 'correlacoes_atributos_treino.csv')
corr_targets.to_csv(SAIDA / 'correlacoes_targets_treino.csv')
pares = []
for i, coluna in enumerate(corr_features.columns):
    for outra in corr_features.columns[i + 1:]:
        valor = corr_features.loc[coluna, outra]
        if abs(valor) > 0.85:
            pares.append({'atributo_1': coluna, 'atributo_2': outra, 'pearson': valor, 'absoluta': abs(valor)})
pares = pd.DataFrame(pares, columns=['atributo_1', 'atributo_2', 'pearson', 'absoluta']).sort_values('absoluta', ascending=False)
pares.to_csv(SAIDA / 'pares_correlacionados_treino.csv', index=False)
print('Pares com |r| > 0.85:', len(pares))
display(pares.head(20))
top = corr_targets.abs().max(axis=1).nlargest(20).index
fig, axes = plt.subplots(1, 2, figsize=(21, 10))
sns.heatmap(corr_features.loc[top, top], cmap='vlag', center=0, vmin=-1, vmax=1, ax=axes[0])
axes[0].set_title('Pearson entre atributos — treino (top 20)')
sns.heatmap(corr_targets.loc[top], annot=True, fmt='.2f', cmap='vlag', center=0, vmin=-1, vmax=1, ax=axes[1])
axes[1].set_title('Pearson com cada target — treino')
plt.tight_layout()
plt.savefig(FIGURAS / 'correlacoes_treino.png', dpi=140, bbox_inches='tight')
plt.show()
for target in TARGETS:
    print(target)
    display(corr_targets[target].sort_values(key=lambda s: s.abs(), ascending=False).head(10).to_frame('pearson'))


## 4 — Pipeline reutilizável

`PreprocessamentoCorrelacao` (definido no módulo importável `svm_audio.py`) valida os nomes,
seleciona apenas os atributos acústicos, trata infinitos, imputa a mediana e elimina
constantes. Ordena os atributos por |correlação com o target| e remove os redundantes
com um atributo já mantido. O limiar 1.0 desativa apenas a remoção por correlação.

Depois aplicamos `StandardScaler` e `LinearSVC(class_weight='balanced')`, como na parte 3.
Não há one-hot: as entradas são todas numéricas. Cada `fit`, inclusive na validação,
aprende suas próprias medianas, seleção e escala. Exportar o pipeline inteiro preserva
essas transformações. O módulo acompanha os modelos para permitir carregamento fora do notebook.


In [ ]:
def criar_pipeline(limiar=1.0, C=1.0):
    return Pipeline([
        ('atributos', PreprocessamentoCorrelacao(features=FEATURES, limiar=limiar)),
        ('scaler', StandardScaler()),
        ('classifier', LinearSVC(
            C=C, class_weight='balanced', random_state=SEED,
            dual='auto', max_iter=50000,
        )),
    ])

# Critério da referência: recall. Desempates: F1, precision e ordem fixa da grade.
# A escolha usa somente validação cruzada do treino, nunca métricas do teste.
def escolher_candidato(resultados_cv):
    ranking = pd.DataFrame(resultados_cv)
    return int(ranking.sort_values(
        ['mean_test_recall', 'mean_test_f1', 'mean_test_precision'],
        ascending=False, kind='stable',
    ).index[0])

GRADE = {
    'atributos__limiar': [1.0, 0.95, 0.85],
    'classifier__C': [0.1, 0.5, 1.0, 2.0],
}
SCORING = {'recall': 'recall', 'precision': 'precision', 'f1': 'f1', 'balanced_accuracy': 'balanced_accuracy'}


## 5 — Treinamento sequencial: cansaço, falta de ar e doença respiratória

Para cada target, treinamos o SVM base e buscamos 12 combinações de `C` e limiar de
correlação, com quatro folds por consulta. O melhor candidato é reajustado nas 16 consultas
de treino pelo `GridSearchCV`. A comparação base/tuning usa o mesmo teste somente depois
de concluir a seleção. O modelo exportado será o escolhido pela validação, independentemente
de vencer ou perder para o base no teste.

As métricas da busca e da avaliação são por áudio inteiro. Cada consulta contribui
com uma única amostra, independentemente de sua duração.


In [ ]:
bases, buscas, modelos = {}, {}, {}
for target in TARGETS:
    print(f'\nTreinando target: {target}')
    y_train = Y_train[target]
    bases[target] = criar_pipeline().fit(X_train, y_train)
    busca = GridSearchCV(
        criar_pipeline(), param_grid=GRADE, scoring=SCORING,
        refit=escolher_candidato, cv=cv, n_jobs=1, error_score='raise',
        return_train_score=False,
    )
    busca.fit(X_train, y_train)
    buscas[target] = busca
    modelos[target] = busca.best_estimator_
    tabela = pd.DataFrame(busca.cv_results_)
    tabela.to_csv(SAIDA / f'validacao_cruzada_{target}.csv', index=False)
    display(tabela[['params', 'mean_test_recall', 'std_test_recall', 'mean_test_f1', 'mean_test_precision']].sort_values(
        ['mean_test_recall', 'mean_test_f1', 'mean_test_precision'], ascending=False
    ).head(5))
    print('Parâmetros escolhidos:', busca.best_params_)
    print('Atributos mantidos:', len(modelos[target].named_steps['atributos'].selecionadas_))


## 6 — Avaliação no teste reservado

Registramos accuracy, balanced accuracy, precision, recall, F1, ROC AUC e matriz de
confusão. A ROC AUC usa `decision_function`, que retorna uma **margem, não probabilidade**.
A decisão binária usa margem > 0 diretamente sobre os atributos do áudio inteiro.

As quatro consultas de teste incluem apenas **uma consulta positiva por target**.
Portanto, recall por consulta pode variar de 0 a 1 com um único erro; os resultados são
exploratórios.


In [ ]:
def metricas(y, pred, scores):
    matriz = confusion_matrix(y, pred, labels=[0, 1])
    tn, fp, fn, tp = matriz.ravel()
    return {
        'accuracy': accuracy_score(y, pred),
        'balanced_accuracy': balanced_accuracy_score(y, pred),
        'precision': precision_score(y, pred, zero_division=0),
        'recall': recall_score(y, pred, zero_division=0),
        'f1': f1_score(y, pred, zero_division=0),
        'roc_auc': roc_auc_score(y, scores),
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
        'n': len(y),
    }

resultados, predicoes = [], []
matrizes = {}
for target in TARGETS:
    for nome, modelo in [('base', bases[target]), ('tuning', modelos[target])]:
        scores = modelo.decision_function(X_test)
        pred = modelo.predict(X_test)
        assert np.array_equal(pred, (scores > 0).astype(int))
        resultados.append({'target': target, 'modelo': nome, 'unidade': 'consulta', **metricas(Y_test[target], pred, scores)})
        detalhe = dados.loc[mascara_teste, METADADOS].reset_index(drop=True).copy()
        detalhe['target'], detalhe['modelo'] = target, nome
        detalhe['real'], detalhe['predito'], detalhe['margem'] = Y_test[target], pred, scores
        predicoes.append(detalhe)
        if nome == 'tuning':
            matrizes[target] = confusion_matrix(Y_test[target], pred, labels=[0, 1])
            print(f'\n{target} — SVM com tuning — áudios inteiros')
            print(classification_report(Y_test[target], pred, labels=[0, 1], zero_division=0))
resultados_df = pd.DataFrame(resultados)
resultados_df.to_csv(SAIDA / 'metricas_teste.csv', index=False)
pd.concat(predicoes, ignore_index=True).to_csv(SAIDA / 'predicoes_teste_consultas.csv', index=False)
display(resultados_df.round(4))
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for j, target in enumerate(TARGETS):
    sns.heatmap(matrizes[target], annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[j])
    axes[j].set(title=f'{target} — áudio inteiro', xlabel='Previsto (0 / 1)', ylabel='Real (0 / 1)')
plt.tight_layout()
plt.savefig(FIGURAS / 'matrizes_confusao.png', dpi=140)
plt.show()


## 7 — Exportação dos três modelos e rastreabilidade

Cada `.joblib` contém o pipeline completo ajustado **apenas no treino**, exatamente o
modelo avaliado acima. Não reajustamos na base completa após o teste. Salvamos o esquema
de entrada, atributos mantidos/removidos, parâmetros, métricas, divisão, hash do Parquet,
versões de bibliotecas e o módulo do transformador. Não é necessário fornecer targets na inferência.

Ao mover os modelos para outra aplicação, leve a pasta `modelos` inteira e instale
`requirements_inferencia.txt` em Python da mesma versão principal/secundária. Os novos áudios
precisam passar pelo mesmo extrator eGeMAPSv02/Functionals usando o áudio inteiro, sem segmentação.
Os `.joblib` devem ser carregados apenas de fontes confiáveis.


In [ ]:
pacotes = ['numpy', 'pandas', 'scipy', 'scikit-learn', 'joblib', 'pyarrow']
versoes = {nome: importlib.metadata.version(nome) for nome in pacotes}
(MODELOS / 'requirements_inferencia.txt').write_text(
    '\n'.join(f'{nome}=={versao}' for nome, versao in versoes.items()) + '\n', encoding='utf-8'
)
shutil.copy2(RAIZ_AUDIO / 'svm_audio.py', MODELOS / 'svm_audio.py')
manifesto = {
    'python': platform.python_version(), 'versoes': versoes, 'seed': SEED,
    'parquet': PARQUET.name, 'sha256_parquet': hashlib.sha256(PARQUET.read_bytes()).hexdigest(),
    'features_entrada': FEATURES, 'targets': TARGETS, 'metadados_excluidos': METADADOS,
    'extrator': 'openSMILE eGeMAPSv02 / Functionals; áudio mono por média dos canais',
    'maior_audio_observado_s': float((dados['fim_s'] - dados['inicio_s']).max()),
    'consultas_treino': sorted(arquivos_treino.tolist()),
    'consultas_teste': sorted(arquivos_teste.tolist()),
    'unidade_entrada': 'audio_inteiro',
    'duracao_segmento_s': None,
    'regra_consulta': 'margem do áudio inteiro > 0',
    'selecao': 'recall de validação; desempate por F1 e precision; teste não seleciona modelo',
    'ajuste_final': 'somente consultas de treino; sem refit na base completa',
    'modelos': {},
}
for target, modelo in modelos.items():
    arquivo_modelo = MODELOS / f'svm_{target}.joblib'
    joblib.dump(modelo, arquivo_modelo)
    seletor = modelo.named_steps['atributos']
    pd.Series(seletor.selecionadas_, name='atributo').to_csv(SAIDA / f'atributos_selecionados_{target}.csv', index=False)
    pd.DataFrame(seletor.removidas_, columns=['removida', 'mantida', 'correlacao_absoluta']).to_csv(
        SAIDA / f'atributos_redundantes_{target}.csv', index=False
    )
    melhor = pd.DataFrame(buscas[target].cv_results_).iloc[buscas[target].best_index_]
    manifesto['modelos'][target] = {
        'arquivo': arquivo_modelo.name,
        'parametros': buscas[target].best_params_,
        'recall_cv': float(melhor['mean_test_recall']),
        'desvio_recall_cv': float(melhor['std_test_recall']),
        'atributos_selecionados': seletor.selecionadas_,
        'atributos_constantes': seletor.constantes_,
        'metricas_teste': resultados_df.query('target == @target and modelo == "tuning"').to_dict(orient='records'),
        'sha256': hashlib.sha256(arquivo_modelo.read_bytes()).hexdigest(),
    }
(MODELOS / 'manifesto.json').write_text(json.dumps(manifesto, indent=2, ensure_ascii=False), encoding='utf-8')
print('Modelos exportados:')
for arquivo in sorted(MODELOS.glob('*.joblib')):
    print(arquivo)


## 8 — Recarga e uso posterior

Na nova aplicação, adicione `modelos` ao caminho de importação para disponibilizar
`svm_audio.py`, carregue o pipeline e chame `predict` com um DataFrame de atributos.
A ordem de entrada pode variar: o transformador reordena pelos nomes e rejeita atributos
obrigatórios ausentes. Colunas extras são ignoradas. Os três modelos são independentes;
um áudio futuro pode receber mais de um rótulo positivo.

A célula usa o teste já reservado para verificar a persistência, sem novo treinamento.
Para dados novos, substitua `novos_atributos` pela leitura do Parquet gerado pelo extrator.


In [ ]:
sys.path.insert(0, str(MODELOS))
novos_atributos = X_test.copy()  # Apenas features; nenhum target é necessário.
previsoes = pd.DataFrame(index=novos_atributos.index)
for target in TARGETS:
    recarregado = joblib.load(MODELOS / f'svm_{target}.joblib')
    previsoes[target] = recarregado.predict(novos_atributos)
    previsoes[f'margem_{target}'] = recarregado.decision_function(novos_atributos)
    np.testing.assert_array_equal(previsoes[target], modelos[target].predict(X_test))
    np.testing.assert_allclose(previsoes[f'margem_{target}'], modelos[target].decision_function(X_test))
print('Recarga validada: classes e margens idênticas para os três modelos.')
display(previsoes.head())


## Conclusão e limites

A tabela abaixo resume os resultados efetivamente obtidos; não reutiliza números da parte 3.
A interpretação deve considerar apenas 20 consultas e a seleção manual de quatro grupos.
Mais consultas, identificação dos participantes e validação em uma base externa são necessárias
para estimar generalização. Correlações acústicas podem refletir locutor, gravação ou contexto.
O grupo chamado `doenca_respiratoria` foi rotulado a partir de tosse; o modelo não verifica doença.

O teste não orientou a escolha de hiperparâmetros. Novas decisões tomadas com base nesses
resultados exigem outro teste independente. Consulte o manifesto e os CSVs para auditar o processo.

### Referências técnicas

- Notebook de referência: `trabalho/trabalho_parte_3.ipynb` (SVM base, tuning por recall e métricas).
- [scikit-learn: prevenção de vazamento com pipelines](https://scikit-learn.org/stable/common_pitfalls.html).
- [scikit-learn: validação cruzada com grupos](https://scikit-learn.org/stable/modules/cross_validation.html).
- [scikit-learn: persistência e compatibilidade de versões](https://scikit-learn.org/stable/model_persistence.html).


In [ ]:
resumo = resultados_df.query('modelo == "tuning"')[
    ['target', 'unidade', 'precision', 'recall', 'f1', 'balanced_accuracy', 'fn', 'fp', 'n']
].copy()
display(resumo.round(4))
for target in TARGETS:
    info = manifesto['modelos'][target]
    print(f"{target}: {len(info['atributos_selecionados'])}/88 atributos; "
          f"recall CV={info['recall_cv']:.3f} ± {info['desvio_recall_cv']:.3f}; "
          f"parâmetros={info['parametros']}")
